In [2]:
from neo4j import GraphDatabase
from neo4j_graphrag.indexes import create_vector_index

URI = "neo4j://10.0.40.100:7687"
AUTH = ("neo4j", "sacf_sacf")

INDEX_NAME = "probill"

# Connect to Neo4j database
driver = GraphDatabase.driver(URI, auth=AUTH)

# Creating the index
create_vector_index(
    driver,
    INDEX_NAME,
    label="Document",
    embedding_property="vectorProperty",
    dimensions=1536,
    similarity_fn="euclidean",
)

In [ ]:
from neo4j_graphrag.indexes import upsert_vectors
from neo4j_graphrag.types import EntityType
INDEX_NAME = "vector-index-name"

# Connect to Neo4j database
driver = GraphDatabase.driver(URI, auth=AUTH)

# Creating the index
create_vector_index(
    driver,
    INDEX_NAME,
    label="Document",
    embedding_property="vectorProperty",
    dimensions=1536,
    similarity_fn="euclidean",
)

In [ ]:
from neo4j import GraphDatabase
from neo4j_graphrag.embeddings.ollama import OllamaEmbeddings
from neo4j_graphrag.retrievers import VectorRetriever

INDEX_NAME = "vector-index-name"

# Connect to Neo4j database
driver = GraphDatabase.driver(URI, auth=AUTH)
# Create Embedder object
# Note: An OPENAI_API_KEY environment variable is required here
embedder = OllamaEmbeddings(model="nomic-embed-text", host="http://10.0.40.49:11434")

# Initialize the retriever
retriever = VectorRetriever(driver, INDEX_NAME, embedder)

# Run the similarity search
query_text = "How do I do similarity search in Neo4j?"
response = retriever.search(query_text=query_text, top_k=5)


In [1]:
from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_ext.tools.mcp import SseMcpToolAdapter, mcp_server_tools,StdioServerParams
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.ui import Console
from autogen_core import CancellationToken
from pathlib import Path

mcp_server_path = str("/workspaces/OrchestrAI-autogen/python/packages/probill/src/probill/mco_tools/mcp-neo4j/servers/mcp-neo4j-cypher")
server_params = StdioServerParams(
    command="uv", args=[
        "--directory",
        mcp_server_path,
        "run",
        "mcp-neo4j-cypher",
        "--db-url",
        "bolt://10.0.40.49:7687",
        "--username", "neo4j",
        "--password",
        "sacf_sacf"
    ]
)

tools = await mcp_server_tools(server_params)

In [3]:
import json
tools_json = []
for tool in tools:
    # print(tool.name)
    tool_json = tool.dump_component().model_dump()
    tool_json["label"] = f"ne4j.{tool.name}"
    tool_json["config"]["name"] = f"ne4j.{tool.name}"
    tool_json["config"]["description"] = tool_json["description"]
    tools_json.append(tool_json)

print(json.dumps(tools_json, indent=2))

[
  {
    "provider": "autogen_ext.tools.mcp.StdioMcpToolAdapter",
    "component_type": "tool",
    "version": 1,
    "component_version": 1,
    "description": "Allows you to wrap an MCP tool running over STDIO and make it available to AutoGen.",
    "label": "ne4j.read-neo4j-cypher",
    "config": {
      "server_params": {
        "command": "uv",
        "args": [
          "--directory",
          "/workspaces/OrchestrAI-autogen/python/packages/probill/src/probill/mco_tools/mcp-neo4j/servers/mcp-neo4j-cypher",
          "run",
          "mcp-neo4j-cypher",
          "--db-url",
          "bolt://10.0.40.49:7687",
          "--username",
          "neo4j",
          "--password",
          "sacf_sacf"
        ],
        "encoding": "utf-8",
        "encoding_error_handler": "strict"
      },
      "tool": {
        "name": "read-neo4j-cypher",
        "description": "Execute a Cypher query on the neo4j database",
        "inputSchema": {
          "type": "object",
          "prop

In [1]:
import json
temp_json = {
  "nodes": [
    {
      "identifier": "Patient",
      "labels": ["Person"],
      "properties": {
        "age": 67,
        "sex": "Male"
      }
    },
    {
      "identifier": "Disease",
      "labels": ["LungCancer"],
      "properties": {
        "type": "NSCLC",
        "stage": "Stage IIIA"
      }
    },
    {
      "identifier": "Symptom",
      "labels": ["Symptom"],
      "properties": {
        "name": "Cough"
      }
    },
    {
      "identifier": "Biopsy",
      "labels": ["Biopsy"],
      "properties": {
        "result": "NSCLC Adenocarcinoma"
      }
    },
    {
      "identifier": "Mutation",
      "labels": ["Mutation"],
      "properties": {
        "type": "EGFR",
        "status": "Negative"
      }
    },
    {
      "identifier": "Treatment",
      "labels": ["Chemotherapy"],
      "properties": {
        "name": "Cisplatin + Pemetrexed"
      }
    },
    {
      "identifier": "DiagnosticTest",
      "labels": ["DiagnosticTest"],
      "properties": {
        "name": "CT Scan",
        "finding": "Right upper lobe mass"
      }
    }
  ],
  "relationships": [
    {
      "startNode": "Patient",
      "type": "HAS_CONDITION",
      "endNode": "Disease"
    },
    {
      "startNode": "Disease",
      "type": "PRESENTS_WITH",
      "endNode": "Symptom"
    },
    {
      "startNode": "Disease",
      "type": "DIAGNOSED_BY",
      "endNode": "Biopsy"
    },
    {
      "startNode": "Biopsy",
      "type": "HAS_MUTATION",
      "endNode": "Mutation"
    },
    {
      "startNode": "Disease",
      "type": "TREATED_WITH",
      "endNode": "Treatment"
    },
    {
      "startNode": "Patient",
      "type": "UNDERGOES",
      "endNode": "DiagnosticTest"
    }
  ]
}
print(temp_json)

{'nodes': [{'identifier': 'Patient', 'labels': ['Person'], 'properties': {'age': 67, 'sex': 'Male'}}, {'identifier': 'Disease', 'labels': ['LungCancer'], 'properties': {'type': 'NSCLC', 'stage': 'Stage IIIA'}}, {'identifier': 'Symptom', 'labels': ['Symptom'], 'properties': {'name': 'Cough'}}, {'identifier': 'Biopsy', 'labels': ['Biopsy'], 'properties': {'result': 'NSCLC Adenocarcinoma'}}, {'identifier': 'Mutation', 'labels': ['Mutation'], 'properties': {'type': 'EGFR', 'status': 'Negative'}}, {'identifier': 'Treatment', 'labels': ['Chemotherapy'], 'properties': {'name': 'Cisplatin + Pemetrexed'}}, {'identifier': 'DiagnosticTest', 'labels': ['DiagnosticTest'], 'properties': {'name': 'CT Scan', 'finding': 'Right upper lobe mass'}}], 'relationships': [{'startNode': 'Patient', 'type': 'HAS_CONDITION', 'endNode': 'Disease'}, {'startNode': 'Disease', 'type': 'PRESENTS_WITH', 'endNode': 'Symptom'}, {'startNode': 'Disease', 'type': 'DIAGNOSED_BY', 'endNode': 'Biopsy'}, {'startNode': 'Biopsy', 